# Friedman Testi
Friedman Testi, Repeated Measures ANOVA'nın (Konu, tekrarlı ölçüm) 
normallik/sphericity sağlanmadığındaki non-parametrik karşılığıdır. Bu, 
Non-Parametrik Testler bölümünün **son ve en karmaşık** üyesi.

## Ne Zaman Kullanılır?
**Aynı bireylerin, 3 veya daha fazla kez** ölçüldüğü durumlarda (örn: 
aynı 20 müşterinin, 4 farklı ay boyunca memnuniyeti), ama normallik 
sağlanmadığında.

## Mantığı
Her **birey için ayrı ayrı**, o bireyin farklı zaman noktalarındaki 
değerlerini **kendi içinde sıralar** (rank verir) — yani her kişi kendi 
4 ölçümü arasında 1'den 4'e kadar sıralanır. Sonra her zaman noktasının 
(örn: "Ocak") tüm bireylerdeki **ortalama sıralamasına** bakılır.

**Sezgisel anlamı:** Eğer bir zaman noktası (örn: Nisan), çoğu bireyde 
sistematik olarak "en yüksek" sırada yer alıyorsa, bu, o zaman noktasının 
gerçekten farklı olduğuna işaret eder.

## Hipotezler
- **H0:** Tüm zaman noktalarının/koşulların dağılımları aynıdır
- **H1:** En az bir zaman noktası/koşul, diğerlerinden farklıdır

## Python'da Kullanımı
```python
from scipy.stats import friedmanchisquare

istatistik, p_degeri = friedmanchisquare(zaman1, zaman2, zaman3, zaman4)
```
Dikkat: Veriler burada **geniş format** halinde olmalı (her zaman noktası 
ayrı bir dizi/liste, `pingouin`'in `rm_anova`'sındaki gibi uzun format 
değil) — kütüphaneler arası bu farklılığa dikkat etmek gerekiyor.

## Post-Hoc'u
Anlamlı çıkarsa, non-parametrik post-hoc olarak **Nemenyi Testi** veya 
Wilcoxon testinin Bonferroni düzeltmeli tekrarlı uygulaması kullanılabilir.

## Bölümün Özeti (Parametrik-Non-Parametrik Eşleşmesi Tamamlandı)
| Parametrik | Non-Parametrik |
|---|---|
| Tek Örneklem T Testi | İşaret Testi |
| Bağımsız İki Örneklem T Testi | Mann-Whitney U |
| Bağımlı Örneklem T Testi | Wilcoxon |
| One-Way ANOVA | Kruskal-Wallis |
| Repeated Measures ANOVA | **Friedman** |

In [ ]:
import numpy as np
import pandas as pd
import pingouin as pg

# İnsanların 1,2,3 ve 4. aylarındaki yaşam memnuniyetleri üzerinden tekrarlı ölçümler ANOVA testini uygulayacağız.
ilk_ay = np.random.exponential(scale=5, size=100)
ikinci_ay = np.random.exponential(scale=8, size=100)
ucuncu_ay = np.random.exponential(scale=11, size=100)
dorduncu_ay = np.random.exponential(scale=14, size=100)
# H0: Tüm zaman noktalarının/koşulların dağılımları aynıdır
# H1: En az bir zaman noktası/koşul, diğerlerinden farklıdır.

# öncelikle her bir gruba normallik testi uygulayalım.
# bundan önce veriyi dataframe formatına getirelim.
kisi_no = np.arange(0,100)
veri = pd.DataFrame({
    'Kişi No': kisi_no,
    'İlk Ay': ilk_ay,
    'İkinci Ay': ikinci_ay,
    'Üçüncü Ay': ucuncu_ay,
    'Dördüncü Ay': dorduncu_ay
})
uzun_format = pd.melt(frame=veri, id_vars='Kişi No', value_vars=['İlk Ay', 'İkinci Ay', 'Üçüncü Ay', 'Dördüncü Ay'], var_name='Aylar', value_name='Değer')
uzun_format

,Kişi No,Aylar,Değer
0,0,İlk Ay,5.203702
1,1,İlk Ay,2.461711
2,2,İlk Ay,1.469463
3,3,İlk Ay,5.709321
4,4,İlk Ay,3.441579
...,...,...,...
395,95,Dördüncü Ay,16.144041
396,96,Dördüncü Ay,7.642602
397,97,Dördüncü Ay,2.973106
398,98,Dördüncü Ay,9.038216


In [ ]:
# Normallik testleri
normallik_sonuclari = pg.normality(data=uzun_format, dv='Değer', group='Aylar')
print(normallik_sonuclari)

                    W          pval  normal
Aylar                                      
İlk Ay       0.761237  1.875289e-11   False
İkinci Ay    0.858999  2.587487e-08   False
Üçüncü Ay    0.794970  1.740080e-10   False
Dördüncü Ay  0.828743  2.095984e-09   False


4 ay da normal dağılmıyor bunu yukarıda yaptığımız normallik testiyle kanıtladık. Bu yüzden aşağıda Repeated Measures ANOVA'nın parametrik olmayan versiyonu Friedman testini uygulayacağız.

In [5]:
# H0: Tüm zaman noktalarının dağılımı aynıdır.
# H1: En az bir zaman noktası, diğer zaman noktalarından farklıdır.
friedman = pg.friedman(data=uzun_format, dv='Değer', within='Aylar', subject='Kişi No')
print(friedman)

         Source       W  ddof1      Q     p_unc
Friedman  Aylar  0.0758      3  22.74  0.000046


In [6]:
ilk_ay_ortalama = np.mean(veri['İlk Ay'])
ikinci_ay_ortalama = np.mean(veri['İkinci Ay'])
ucuncu_ay_ortalama = np.mean(veri['Üçüncü Ay'])
dorduncu_ay_ortalama = np.mean(veri['Dördüncü Ay'])
print(f'1.aya ait yaşam memnuniyeti puanlarının ortalaması: {ilk_ay_ortalama}')
print(f'2.aya ait yaşam memnuniyeti puanlarının ortalaması: {ikinci_ay_ortalama}')
print(f'3.aya ait yaşam memnuniyeti puanlarının ortalaması: {ucuncu_ay_ortalama}')
print(f'4.aya ait yaşam memnuniyeti puanlarının ortalaması: {dorduncu_ay_ortalama}')

1.aya ait yaşam memnuniyeti puanlarının ortalaması: 5.172713552909333
2.aya ait yaşam memnuniyeti puanlarının ortalaması: 7.971624340791524
3.aya ait yaşam memnuniyeti puanlarının ortalaması: 11.68182846952087
4.aya ait yaşam memnuniyeti puanlarının ortalaması: 12.477539856908349


Friedman testi sonuçlarına göre, en az bir zaman noktasının dağılımı (sıralaması), diğer zaman noktalarının dağılımından anlamlı şekilde 
farklıdır (Q=22.74, p<0.001). Betimsel olarak aylara ait ortalamalara bakıldığında (ki bu Friedman'ın test ettiği şey değil ama betimsel bir gözlem olarak), zaman ilerledikçe puanların tutarlı bir artış eğilimi gösterdiği görülmektedir (1. ay: 5.17, 2. ay: 7.97, 3. ay: 11.68, 4. ay: 12.48). Bu doğrultuda, zaman içinde puanlarda anlamlı bir artış olduğu söylenebilir.